# LOO & Generalized Cross-Validation

Leave-one-out evaluates one excluded observation at a time. In least squares, a leverage correction recovers those predictions from a single fit. Generalized cross-validation replaces individual leverages with their average.

:::{admonition} Two leverage corrections
:class: important

LOO corrects each residual using $1-h_{ii}$. GCV replaces individual leverage with the average, $\operatorname{df}/n$.
:::

## Leave-One-Out Cross-Validation

With $k=n$, each fold contains one observation:

$$\widehat{\operatorname{Err}}_{\mathrm{LOO}}
=\frac1n\sum_{i=1}^nL(Y_i,\widehat f^{(-i)}(X_i)).$$

A direct implementation fits $n$ models on $n-1$ observations each. For a fixed algorithm under iid sampling, its expected score is $\operatorname{Err}_{n-1}$. This is often close to $\operatorname{Err}_n$ when the learning curve changes slowly; it is not exactly unbiased for the full-data model's error.

## The OLS Shortcut

Let $X$ be a full-rank $n\times p$ fixed design, with rows $x_i^\top$. Set

$$H=X(X^\top X)^{-1}X^\top,\qquad
\widehat Y=HY,\qquad e_i=Y_i-\widehat Y_i,\qquad h_{ii}=H_{ii}.$$

If deleting observation $i$ leaves a full-rank design, then $h_{ii}<1$ and

$$\boxed{Y_i-\widehat f^{(-i)}(X_i)=\frac{e_i}{1-h_{ii}}.}$$

Consequently, for **squared loss**,

$$\boxed{\widehat{\operatorname{Err}}_{\mathrm{LOO}}
=\frac1n\sum_i\left(\frac{e_i}{1-h_{ii}}\right)^2.}$$

The numerator uses ordinary full-fit residuals; the denominator corrects for using the observation in its own fit. $h_{ii}$ is **leverage**, determined by the design. High leverage can amplify a small residual, but leverage alone is not a complete measure of influence.

:::{dropdown} Derivation via Sherman-Morrison
Write $M=X^\top X$, $h_i=x_i^\top M^{-1}x_i$, and $e_i=Y_i-x_i^\top\widehat\beta$. Removing row $i$ gives

$$X_{-i}^\top X_{-i}=M-x_ix_i^\top,\qquad
X_{-i}^\top Y_{-i}=X^\top Y-x_iY_i.$$

Sherman-Morrison yields

$$(M-x_ix_i^\top)^{-1}=M^{-1}+
\frac{M^{-1}x_ix_i^\top M^{-1}}{1-h_i}.$$

Multiplying by $X^\top Y-x_iY_i$ and collecting terms gives

$$\widehat\beta^{(-i)}=\widehat\beta-\frac{M^{-1}x_i e_i}{1-h_i}.$$

Thus

$$\widehat f^{(-i)}(X_i)=\widehat Y_i-\frac{h_i e_i}{1-h_i},$$

and

$$Y_i-\widehat f^{(-i)}(X_i)
=e_i+\frac{h_i e_i}{1-h_i}=\frac{e_i}{1-h_i}.$$

Squaring and averaging gives the shortcut.
:::

### Scope of the Formula

The identity is exact for the specified OLS refits, and for certain penalized least-squares smoothers with the appropriate fixed penalty convention. Merely writing $\widehat Y=HY$ for an arbitrary full-data linear map is **not sufficient**: the rule after deletion must have the required relationship to the full fit.

It also does not automatically reproduce a pipeline that repeats feature selection, changes the basis, or retunes a penalty after each deletion. If $h_{ii}=1$, the displayed OLS shortcut is undefined and the deleted design loses rank.

## Worked Leverage Example

Use $x=(-1,0,1)$ and $y=(1,0,1)$, fitting an intercept and slope. The fitted line is constant at $2/3$:

$$e=(1/3,-2/3,1/3),\qquad h=(5/6,1/3,5/6).$$

| Observation | Training residual $e_i$ | $1-h_{ii}$ | LOO residual |
|---|---:|---:|---:|
| $x=-1$ | $1/3$ | $1/6$ | $2$ |
| $x=0$ | $-2/3$ | $2/3$ | $-1$ |
| $x=1$ | $1/3$ | $1/6$ | $2$ |

The training MSE is $2/9\approx0.2222$, but

$$\widehat{\operatorname{Err}}_{\mathrm{LOO}}=(4+1+4)/3=3.$$

Deleting an endpoint leaves a line fitted through the other two points. Predicting the excluded endpoint is extrapolation, which explains the large LOO errors despite modest training residuals.

## Generalized Cross-Validation

::::{tab-set}

:::{tab-item} LOO: individual correction

$$\operatorname{LOO}=\frac1n\sum_i\left(\frac{e_i}{1-h_{ii}}\right)^2.$$

Uses each observation’s leverage. Exact for the specified OLS deleted fits.
:::

:::{tab-item} GCV: average correction

$$\operatorname{GCV}=\frac{\operatorname{RSS}/n}{(1-\operatorname{df}/n)^2}.$$

Uses one common correction. Efficient, but generally an approximation to LOO.
:::
::::

Replace each $h_{ii}$ in the LOO correction by the average leverage

$$\bar h=\frac1n\sum_i h_{ii}=\frac{\operatorname{tr}(H)}n.$$

The **GCV score** is

$$\boxed{\operatorname{GCV}
=\frac1n\sum_i\left(\frac{e_i}{1-\operatorname{tr}(H)/n}\right)^2
=\frac{\operatorname{RSS}/n}{[1-\operatorname{df}/n]^2}.}$$

For OLS, $\operatorname{df}=p$. Compute the trace without forming the full $n\times n$ hat matrix. The same idea applies to regularized smoothers with noninteger effective degrees of freedom.

| LOO | GCV |
|---|---|
| Corrects each residual using its own leverage. | Uses a common average-leverage correction. |
| Exact OLS deleted-fit score when the conditions hold. | Approximation to LOO, not literal leave-one-out refitting. |
| Can be dominated by individual high-leverage cases. | Removes individual leverage denominators, but may hide their effect. |

For the three-point example, $\operatorname{df}=2$ and

$$\operatorname{GCV}=\frac{2/9}{(1-2/3)^2}=2,$$

which differs from LOO's value of $3$. If all leverages are equal, the two formulas agree. When df approaches $n$, the GCV denominator becomes small; at df $=n$ the score is undefined.

:::{dropdown} Why GCV Is Invariant to Orthogonal Rotation
For a fixed linear smoother, transform the observation coordinates using $Q^\top Q=QQ^\top=I$:

$$Y^*=QY,\qquad H^*=QHQ^\top,\qquad e^*=Qe.$$

Then

$$\|e^*\|^2=\|e\|^2,\qquad
\operatorname{tr}(H^*)=\operatorname{tr}(H).$$

Both parts of GCV are unchanged. Individual diagonal entries of $H^*$ need not equal those of $H$, so the corresponding coordinate-wise LOO expression can change. For OLS, this smoother transformation follows from $X^*=QX$.

This is an algebraic motivation for GCV. Rotating rows mixes observations and changes what it means to omit one observation; it does not make the original LOO assessment invalid.
:::

:::{admonition} Connection Back to Covariance Penalties
:class: note

For $a=\operatorname{df}/n$ small,

$$(1-a)^{-2}=1+2a+O(a^2).$$

Therefore

$$\operatorname{GCV}\approx\operatorname{err}_{\mathrm{train}}
+\frac{2\operatorname{err}_{\mathrm{train}}\operatorname{df}}n.$$

This resembles the covariance correction $\operatorname{err}_{\mathrm{train}}+2\sigma^2\operatorname{df}/n$, with training MSE playing the role of a rough noise estimate. The resemblance is a first-order approximation, not an exact identity.
:::

## A Subtle Point: Estimating the Error of This Fit

In correctly specified fixed-design Gaussian OLS, $\widehat\beta$ is independent of $e=(I-H)Y$. The LOO score with fixed leverages is a function of $e$, while the fitted coefficients depend on the projected component $HY$.

This explains an important distinction: an error score need not track how fortunate or unfortunate this particular coefficient estimate was. Estimating **average performance** across training samples is different from closely predicting the conditional error of one realized fit.